# Plots for First Hit Analysis

The plots created in this notebook aim to investigate the use of the first hit in a track to initialise the dynamic queries.

In [2]:
import sys                                                                                                                                                                                                         
sys.path.insert(0, "/share/rcif2/mmangat/hepattn/src/hepattn/experiments/trackml/eval/")  

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from track_evaluate import load_events

In [3]:
training_colours = {
    "first_hit": "tab:orange",
    "last_hit": "tab:blue"
}

tracking_fnames = {
    "first_hit": "/share/rcif2/mmangat/logs/TRK-v8-3l-fast-fix-vec_20260316-T104827/ckpts/epoch=020-val_loss=1.48183_test_eval.h5",
    "last_hit": "/share/rcif2/mmangat/logs/TRK-v8-3l-fast-fix-vec_20260428-T172805/ckpts/epoch=004-val_loss=0.58669_test_eval.h5",
}

tracking_input = "/share/rcif2/pduckett/data/prepped/test/"

In [ ]:
def load_vertex_positions(h5_path, data_dir):
    """Load vx, vy, vz for each particle slot by joining H5 particle_ids with parquet files.

    Returns a DataFrame indexed the same way as parts from load_events (event_id, particle slot order).
    """
    rows = []
    with h5py.File(h5_path, "r") as f:
        for event_id in f.keys():
            particle_ids = np.array(f[f"{event_id}/targets/particle_id"][0])  # (N_slots,)

            parquet_path = Path(data_dir) / f"event0000{event_id}-parts.parquet"
            parts_df = pd.read_parquet(parquet_path, columns=["particle_id", "vx", "vy", "vz"])
            id_to_row = parts_df.set_index("particle_id")

            for pid in particle_ids:
                if pid in id_to_row.index:
                    row = id_to_row.loc[pid]
                    rows.append({"event_id": event_id, "vx": row["vx"], "vy": row["vy"], "vz": row["vz"]})
                else:
                    rows.append({"event_id": event_id, "vx": np.nan, "vy": np.nan, "vz": np.nan})

    return pd.DataFrame(rows)


# Load vertex positions for each model's H5 file
import h5py
from pathlib import Path

vertex_data = {
    name: load_vertex_positions(fname, tracking_input)
    for name, fname in tracking_fnames.items()
}

FileNotFoundError: [Errno 2] No such file or directory: '/share/rcif2/pduckett/data/prepped/test/event29800-parts.parquet'